In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark pyspark

In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CustomerChurnPrediction").getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
df = spark.read.csv("WA_Fn-UseC_-Telco-Customer-Churn.csv",header=True,inferSchema=True)

df.printSchema()

df.show(5)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

+----------+------+-------------+-------+----------+------+------------+---------

In [ ]:
from pyspark.sql.functions import col

df = df.drop("customerID")

df = df.withColumn("TotalCharges",col("TotalCharges").cast("double"))

df = df.na.drop()

print("Total Rows:", df.count())


Total Rows: 7032


In [ ]:
df.groupBy("Churn").count().show()

df.groupBy("Contract").count().show()

df.groupBy("Churn").avg("MonthlyCharges").show()

+-----+-----+
|Churn|count|
+-----+-----+
|   No| 5163|
|  Yes| 1869|
+-----+-----+

+--------------+-----+
|      Contract|count|
+--------------+-----+
|Month-to-month| 3875|
|      One year| 1472|
|      Two year| 1685|
+--------------+-----+

+-----+-------------------+
|Churn|avg(MonthlyCharges)|
+-----+-------------------+
|   No|  61.30740848343966|
|  Yes|   74.4413322632423|
+-----+-------------------+



In [ ]:
from pyspark.ml.feature import StringIndexer

categorical_columns = ['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Churn']

indexers = []

for column in categorical_columns:
    indexer = StringIndexer(inputCol=column,outputCol=column + "_Index")
    indexers.append(indexer)

In [ ]:
from pyspark.ml.feature import VectorAssembler

numeric_columns = [
    'SeniorCitizen',
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

indexed_columns = [col + "_Index" for col in categorical_columns[:-1]]

feature_columns = numeric_columns + indexed_columns

assembler = VectorAssembler(inputCols=feature_columns,outputCol="features")

In [ ]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + [assembler])

pipeline_model = pipeline.fit(df)

processed_df = pipeline_model.transform(df)

final_data = processed_df.select("features",col("Churn_Index").alias("label"))

train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print("Training Rows:", train_data.count())
print("Testing Rows:", test_data.count())

Training Rows: 5690
Testing Rows: 1342


In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features',labelCol='label')

lr_model = lr.fit(train_data)

In [ ]:
predictions = lr_model.transform(test_data)

predictions.select("label","prediction","probability").show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |1.0       |[0.22116385732218222,0.7788361426778178]|
|0.0  |1.0       |[0.27220347588616695,0.7277965241138331]|
|1.0  |1.0       |[0.2255704448772851,0.7744295551227149] |
|1.0  |1.0       |[0.2274906608157085,0.7725093391842914] |
|1.0  |1.0       |[0.23554954805786482,0.7644504519421351]|
|1.0  |1.0       |[0.22639306679114557,0.7736069332088544]|
|1.0  |1.0       |[0.25255523635081845,0.7474447636491816]|
|0.0  |1.0       |[0.36953602213182013,0.6304639778681799]|
|1.0  |1.0       |[0.2659515137727918,0.7340484862272082] |
|0.0  |1.0       |[0.32686647815109743,0.6731335218489025]|
+-----+----------+----------------------------------------+
only showing top 10 rows



In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="accuracy")

accuracy = accuracy_evaluator.evaluate(predictions)

precision_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedPrecision")

precision = precision_evaluator.evaluate(predictions)

recall_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedRecall")

recall = recall_evaluator.evaluate(predictions)

auc_evaluator = BinaryClassificationEvaluator(labelCol="label")

auc = auc_evaluator.evaluate(predictions)

print("\n========== MODEL PERFORMANCE ==========")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"AUC Score : {auc:.4f}")


========== MODEL PERFORMANCE ==========
Accuracy  : 0.8182
Precision : 0.8112
Recall    : 0.8182
AUC Score : 0.8425


In [ ]:
predictions.groupBy("label","prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|  187|
|  0.0|       1.0|  100|
|  1.0|       0.0|  144|
|  0.0|       0.0|  911|
+-----+----------+-----+



In [ ]:
lr_model.save("/content/churn_model")

print("Model Saved Successfully!")

Model Saved Successfully!


In [ ]:
df.createOrReplaceTempView("customers")

spark.sql("""
SELECT Contract, COUNT(*) AS total_customers
FROM customers
GROUP BY Contract
""").show()

spark.sql("""
SELECT Churn, AVG(MonthlyCharges) AS avg_monthly_charge
FROM customers
GROUP BY Churn
""").show()

+--------------+---------------+
|      Contract|total_customers|
+--------------+---------------+
|Month-to-month|           3875|
|      One year|           1472|
|      Two year|           1685|
+--------------+---------------+

+-----+------------------+
|Churn|avg_monthly_charge|
+-----+------------------+
|   No| 61.30740848343966|
|  Yes|  74.4413322632423|
+-----+------------------+



In [ ]:
spark.stop()